# Spatial Intersection Engine

This notebook is the core of the project. It answers the question: 
when a precinct straddles two congressional districts, what fraction of that precinct's voters belong to each district?

We answer that by overlaying census blocks onto both layers. Each census block sits entirely within one precinct and one district. By summing the population of blocks in each (precinct, district) combination, we get an accurate population-based split, not just a land area split.

The output is a weights table: for every precinct, what percentage of its population falls into each congressional district.

## Step 1 — Load all three layers

Loads the filtered Travis County files saved in notebooks 01 and 02. All three must share the same CRS (coordinate system) — confirmed by the print statements. CRS 3081 is the Texas state plane projection.

In [1]:
import geopandas as gpd
import pandas as pd

# Load all three layers
precincts = gpd.read_file('../data/processed/travis_precincts_2020.gpkg')
districts = gpd.read_file('../data/processed/travis_districts_2026.gpkg')
blocks = gpd.read_file('../data/processed/travis_blocks_2020.gpkg')

print(f"Precincts: {len(precincts)} | CRS: {precincts.crs.to_epsg()}")
print(f"Districts: {len(districts)} | CRS: {districts.crs.to_epsg()}")
print(f"Blocks: {len(blocks)} | CRS: {blocks.crs.to_epsg()}")

Precincts: 247 | CRS: 3081
Districts: 7 | CRS: 3081
Blocks: 16906 | CRS: 3081


## Step 2 — Tag each census block with its precinct

Spatial join: for each census block, find which precinct it falls inside. The result has more rows than blocks (24,747 vs 16,906) because some blocks sit on precinct boundary lines and get assigned to multiple precincts. 
Zero null assignments means every block got matched.

In [2]:
# Step A: Tag each census block with the precinct it falls in
blocks_with_precinct = gpd.sjoin(
    blocks, 
    precincts[['PCTKEY', 'geometry']], 
    how='left', 
    predicate='intersects'
)

print(f"Blocks after precinct join: {len(blocks_with_precinct)}")
print(f"Null precinct assignments: {blocks_with_precinct['PCTKEY'].isna().sum()}")
blocks_with_precinct[['SCTBKEY', 'total', 'PCTKEY']].head(5)

Blocks after precinct join: 24747
Null precinct assignments: 0


,SCTBKEY,total,PCTKEY
0,484530357003010,131,4530374
1,484530357003011,87,4530374
2,484530357003012,133,4530374
3,484530357003013,102,4530374
4,484530357003014,73,4530374


## Step 3 — Check column names before the second join

The first spatial join added an index_right column that will cause an error in the next join if not removed. This cell confirms what columns exist so we can clean up before proceeding.

In [5]:
print(blocks_with_precinct.columns.tolist())

['BG', 'State', 'CNTY', 'TRT', 'BLK', 'SCTBKEY', 'vtd', 'blkkey', 'CTBKEY', 'CNTYKEY', 'Shape_area', 'Shape_len', 'total', 'anglo', 'asian', 'hisp', 'black', 'geometry', 'index_right', 'PCTKEY']


## Step 4 — Tag each census block with its congressional district

Removes the leftover index column from Step 2, then does a second spatial join to tag each block with its congressional district.

Each block now has both a precinct ID and a district ID. This is the key step, it tells us which district each block's population belongs to, which is how we figure out how to split precinct vote totals.

In [6]:
blocks_with_precinct = blocks_with_precinct.drop(columns=['index_right'])

# Step B: Tag each census block with the district it falls in
blocks_with_both = gpd.sjoin(
    blocks_with_precinct,
    districts[['District', 'geometry']],
    how='left',
    predicate='intersects'
)

print(f"Blocks after district join: {len(blocks_with_both)}")
print(f"Null district assignments: {blocks_with_both['District'].isna().sum()}")
blocks_with_both[['SCTBKEY', 'total', 'PCTKEY', 'District']].head(5)

Blocks after district join: 27246
Null district assignments: 0


,SCTBKEY,total,PCTKEY,District
0,484530357003010,131,4530374,11
1,484530357003011,87,4530374,11
2,484530357003012,133,4530374,11
3,484530357003013,102,4530374,11
4,484530357003014,73,4530374,11


## Step 5 — Calculate population weights for each precinct/district fragment

Groups blocks by (precinct, district) combination and sums their population. Then calculates the weight for each fragment:

weight = population in this fragment / total precinct population

Example: if precinct 4530101 has 20,000 people total and 19,778 live in the part that falls in District 37, the weight for that fragment is 19,778 / 20,000 = 0.99. That means 99% of that precinct's votes get allocated to District 37.

418 total fragments across 247 precincts and 7 districts.

In [7]:
# Step C: Group by (precinct, district) and sum population
fragments = blocks_with_both.groupby(['PCTKEY', 'District'])['total'].sum().reset_index()
fragments.columns = ['old_precinct_id', 'new_district_id', 'fragment_population']

# Step D: Calculate weight = fragment_population / total precinct population
precinct_totals = fragments.groupby('old_precinct_id')['fragment_population'].sum()
fragments['precinct_total'] = fragments['old_precinct_id'].map(precinct_totals)
fragments['weight'] = fragments['fragment_population'] / fragments['precinct_total']

# Drop the helper column
fragments = fragments.drop(columns=['precinct_total'])

print(f"Total fragments: {len(fragments)}")
print(fragments.head(10))

Total fragments: 418
  old_precinct_id  new_district_id  fragment_population    weight
0         4530101               27                  139  0.006979
1         4530101               37                19778  0.993021
2         4530102               37                 5854  1.000000
3         4530103               37                 3743  1.000000
4         4530104               37                 5050  1.000000
5         4530105               10                 2369  0.068090
6         4530105               11                  228  0.006553
7         4530105               27                14260  0.409864
8         4530105               37                17935  0.515492
9         4530106               10                 3113  0.152748


## Validation — confirm weights sum to 1.0 per precinct

For every precinct, all its fragment weights must add up to exactly 1.0. If they don't, votes would be lost or double-counted in the interpolation step.

Min and max should both be 1.0 for non-zero precincts. 
The "2 precincts not summing to 1.0" result is expected, those are 
zero-population precincts (parks, commercial areas) with no votes to allocate.

In [8]:
# Validation: weights must sum to 1.0 per precinct
weight_sums = fragments.groupby('old_precinct_id')['weight'].sum()

print(f"Min weight sum: {weight_sums.min():.6f}")
print(f"Max weight sum: {weight_sums.max():.6f}")
print(f"Any precinct not summing to 1.0: {(abs(weight_sums - 1.0) > 0.001).sum()}")

Min weight sum: 0.000000
Max weight sum: 1.000000
Any precinct not summing to 1.0: 2


## Investigate the two precincts not summing to 1.0

Confirms these are zero-population precincts, not errors.
Zero population divided by zero total = NaN weight, which is expected and handled in the next step.

In [9]:
# Find the problematic precincts
problem_precincts = weight_sums[abs(weight_sums - 1.0) > 0.001]
print("Problematic precincts:")
print(problem_precincts)

# Look at their fragments
print("\nTheir fragments:")
print(fragments[fragments['old_precinct_id'].isin(problem_precincts.index)])

Problematic precincts:
old_precinct_id
4530127    0.0
4530213    0.0
Name: weight, dtype: float64

Their fragments:
    old_precinct_id  new_district_id  fragment_population  weight
50          4530127               37                    0     NaN
110         4530213               10                    0     NaN
111         4530213               37                    0     NaN


## Fix zero-population precincts and re-validate

Replaces NaN weights with 0 for zero-population precincts, then re-runs the validation. The two precincts still show 0.0 (not 1.0) which is correct, they have no population so no votes will ever be allocated to them.

In [10]:
# Fill NaN weights with 0 for zero-population precincts
fragments['weight'] = fragments['weight'].fillna(0)

# Re-run validation
weight_sums = fragments.groupby('old_precinct_id')['weight'].sum()
print(f"Min weight sum: {weight_sums.min():.6f}")
print(f"Max weight sum: {weight_sums.max():.6f}")
print(f"Precincts not summing to 1.0 (excluding zero-pop): {(abs(weight_sums - 1.0) > 0.001).sum()}")

# Check for nulls
print(f"Null values in output: {fragments.isnull().sum().sum()}")

Min weight sum: 0.000000
Max weight sum: 1.000000
Precincts not summing to 1.0 (excluding zero-pop): 2
Null values in output: 0


## Final validation — exclude zero-population precincts and confirm

The definitive check. All 245 precincts with actual population must sum to exactly 1.0. If "Any non-zero precinct not summing to 1.0" returns 0, the weights are correct and we can proceed to vote interpolation.

In [11]:
# Improved validation - exclude zero population precincts
precinct_pops = fragments.groupby('old_precinct_id')['fragment_population'].sum()
zero_pop_precincts = precinct_pops[precinct_pops == 0].index

weight_sums = fragments.groupby('old_precinct_id')['weight'].sum()
non_zero_weight_sums = weight_sums[~weight_sums.index.isin(zero_pop_precincts)]

print(f"Zero population precincts: {len(zero_pop_precincts)} (expected, no votes to allocate)")
print(f"Non-zero population precincts: {len(non_zero_weight_sums)}")
print(f"Min weight sum (non-zero precincts): {non_zero_weight_sums.min():.6f}")
print(f"Max weight sum (non-zero precincts): {non_zero_weight_sums.max():.6f}")
print(f"Any non-zero precinct not summing to 1.0: {(abs(non_zero_weight_sums - 1.0) > 0.001).sum()}")

Zero population precincts: 2 (expected, no votes to allocate)
Non-zero population precincts: 245
Min weight sum (non-zero precincts): 1.000000
Max weight sum (non-zero precincts): 1.000000
Any non-zero precinct not summing to 1.0: 0


## Save the population weights

Saves the validated weights table to /data/processed/travis_population_weights.csv.

This is the key output of this notebook and the main input for notebook 04. Each row represents one (precinct, district) fragment with its population weight. 418 rows total covering all 247 Travis County precincts.

Columns:
- old_precinct_id: the 2020 precinct
- new_district_id: the 2026 congressional district
- fragment_population: how many people live in this overlap area
- weight: fraction of the precinct's votes that go to this district

In [12]:
# Save the population weights
fragments.to_csv('../data/processed/travis_population_weights.csv', index=False)

print("Saved to /data/processed/travis_population_weights.csv")
print(f"\nFinal output shape: {fragments.shape}")
print(f"Columns: {list(fragments.columns)}")
print(f"\nSample output:")
print(fragments.head(10))

Saved to /data/processed/travis_population_weights.csv

Final output shape: (418, 4)
Columns: ['old_precinct_id', 'new_district_id', 'fragment_population', 'weight']

Sample output:
  old_precinct_id  new_district_id  fragment_population    weight
0         4530101               27                  139  0.006979
1         4530101               37                19778  0.993021
2         4530102               37                 5854  1.000000
3         4530103               37                 3743  1.000000
4         4530104               37                 5050  1.000000
5         4530105               10                 2369  0.068090
6         4530105               11                  228  0.006553
7         4530105               27                14260  0.409864
8         4530105               37                17935  0.515492
9         4530106               10                 3113  0.152748
